# CS604 Colab launcher — Ana: taxonomy-aware label smoothing (pilot scale)

Class IDs in this dataset follow the taxonomy: neighbouring IDs are close relatives (their prototypes have cosine 0.80 vs 0.28 for random pairs), and the model's mistakes are mostly look-alike relatives. The recipe's label smoothing (0.1) spreads that share of the target evenly over all 10,000 classes. These arms spread it over the true class's **relatives** instead (`src/losses.py`, `neighbour_smoothing`):

| Arm | Smoothing |
|---|---|
| A_uniform_smoothing | 0.1 evenly over all classes (the recipe) |
| ID_neighbour_smoothing | 0.1 onto the 10 classes within 5 IDs (closer weighted more) |
| PROTO_neighbour_smoothing | 0.1 onto the 10 most similar prototypes (no ID order assumed) |
| ID_MIX_half_uniform_half_neighbours | 0.05 evenly + 0.05 onto ID neighbours |
| ID_02_stronger_neighbour_smoothing | 0.2 onto ID neighbours |

Base: LoRA r24 without extras (CodaBench 0.784, 4.12%). The pilot's 1,000 classes are **100 blocks of 10 consecutive IDs** (`class_block: 10`), so each class's relatives are trained alongside it, as in the full dataset. That is a different pilot subset from the earlier ablations, so compare arms with this run's A only.

Results show `final_val_acc` (top-1) and `final_val_acc_near5` (prediction within 5 IDs of the label, roughly "right genus"). `final_val_loss` is not comparable across arms (different targets).

About 50 min on an A100: data ~15 min, prototypes ~5 min, 5 arms × ~6 min. Results go to `CS604_RUNS/ablation_ablation_taxonomy_smoothing/`. Finished arms are skipped if you Run all again.

Runtime: **A100 GPU**. Use **Runtime → Run all**.


In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
METHOD = "lora_r24_336px_qkv-proj-fc_no-extras_proto-cosine_2.4h"  # base config; only used to build prototypes_336.pt if it is missing
PROTO_PT = f"{DRIVE_OUT}/prototypes_336.pt"
SPEC = "configs/lora_r24_336px_no-extras_neighbour-smoothing_1k-class-pilot.yaml"
PILOT_DATA_DIR = "/content/pilot_data"
FEATURE_DIR = "/content/features_pilot"  # local disk
# End the runtime after the last cell, or as soon as training fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 4b. Build class prototypes at 336 (once)

Same as Van's prototype head, but built from 336 features (built with `configs/{METHOD}.yaml`, so at its `img_size`). Run **once**; about 20–25 min on an A100. Takes a while (embeds train images). Skip if the file already exists on Drive.

In [ ]:
from pathlib import Path

if Path(PROTO_PT).is_file():
    print("Already have", PROTO_PT)
else:
    !python scripts/build_prototypes.py --config configs/{METHOD}.yaml --data_root {DATA_DIR} --output {PROTO_PT}

## 5. Train the five arms

In [ ]:
import subprocess
import sys
from pathlib import Path

OUT_DIR = Path(DRIVE_OUT) / f"ablation_{Path(SPEC).stem}"
PILOT_ROOT = f"{PILOT_DATA_DIR}/c1000_h5_s0_b10"  # built by scripts/ablation.py (100 blocks of 10 IDs)


def run(cmd, what):
    proc = subprocess.run([sys.executable, "-u", *cmd])
    if proc.returncode != 0:
        end_runtime(f"{what} failed (exit {proc.returncode}); see the output above")
        raise SystemExit(f"{what} failed")


def ablation(*only):
    cmd = ["scripts/ablation.py", "--spec", SPEC, "--data_root", DATA_DIR, "--output_dir", DRIVE_OUT,
           "--pilot_data_dir", PILOT_DATA_DIR, "--feature_dir", FEATURE_DIR]
    run(cmd + (["--only", *only] if only else []), "ablation")


ablation()


## 6. Results

`delta_vs_A_points`: pilot holdout top-1 against A, in percentage points; treat under ~0.3 as a tie (one seed). `final_val_acc_near5`: share of holdout photos predicted within 5 IDs of the true class.

- **ID vs A**: does smoothing onto relatives beat even smoothing?
- **PROTO vs ID**: is prototype similarity a better guide than the ID order?
- **ID_MIX / ID_02**: how much of the smoothing should go to relatives?
- **near5 − top-1**: how many errors land on a relative (wrong species, right genus).

In [ ]:
import pandas as pd

table = pd.read_csv(OUT_DIR / "ablation.csv")
table["errors_on_relatives"] = ((table["final_val_acc_near5"] - table["final_val_acc"])
                                / (1 - table["final_val_acc"])).round(3)
display(table[["arm", "final_val_acc", "delta_vs_A_points", "final_val_acc_near5", "errors_on_relatives",
               "best_val_acc", "early_dip", "backbone_pct", "training_min", "peak_vram_gb", "wall_min"]])


## 7. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")